In [2]:
!apt-get install openjdk-17-jdk-headless -qq > /dev/null

In [3]:
import numpy as np
from sklearn.datasets import make_blobs
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").getOrCreate()
sc = spark.sparkContext

In [12]:
n = 2000
B = 10  # 클러스터 개수 (blobs)
d = 2
threshold = 5

X, _ = make_blobs(n_samples=n, centers=B, n_features=d, random_state=0)

In [13]:
rdd = sc.parallelize([(i, X[i]) for i in range(n)])

In [14]:
block_size = 500
num_blocks = int(np.ceil(n / block_size))

In [15]:
# 각 점에 블록 번호 부여
def assign_block(t):
    i, x = t
    block_id = i // block_size
    return (block_id, (i, x))

rdd_blocked = rdd.map(assign_block)  # (block_id, (i, x))

In [16]:
# 블록끼리 모든 조합 생성 (b1 <= b2)
blocks = rdd_blocked.groupByKey().mapValues(list).cache()
block_pairs = blocks.cartesian(blocks).filter(lambda t: t[0][0] <= t[1][0])

In [ ]:
# cartesian()은
# RDD A의 모든 원소 x RDD B의 모든 원소의 조합
# ((0, data0), (0, data0))
# ((0, data0), (1, data1))
# ((0, data0), (2, data2))
# ((0, data0), (3, data3))
# ((1, data1), (0, data0))
# ((1, data1), (1, data1))
# ((1, data1), (2, data2))
# ...
# ((3, data3), (3, data3))


In [24]:
# t[0][0] <= t[1][0]을 통해 대칭 쌍 제거
# (0,1)과 (1,0)은 같은 의미니까.
# block 0,1,2,3이 있었다면 이렇게 변함
# ((0, data0), (0, data0))
# ((0, data0), (1, data1))
# ((0, data0), (2, data2))
# ((0, data0), (3, data3))
# ((1, data1), (1, data1))
# ((1, data1), (2, data2))
# ((1, data1), (3, data3))
# ((2, data2), (2, data2))
# ((2, data2), (3, data3))
# ((3, data3), (3, data3))

In [20]:
# 블록 간 조인 수행
def block_self_join(block_pair):
    """block_pair = ((b1, [(i, x_i), ...]), (b2, [(j, x_j), ...]))"""
    (b1, data1), (b2, data2) = block_pair
    result = []
    for i, x in data1:
        for j, y in data2:
            if i < j:  # 중복 제거
                dist = np.sqrt(np.sum((x - y) ** 2))
                if dist <= threshold:
                    result.append(((i, j), dist))
    return result

In [21]:
# 각 블록 쌍에 대해 거리 계산
rdd2 = block_pairs.flatMap(block_self_join)

In [22]:
count = rdd2.count()
print(f"Number of similar pairs (distance ≤ {threshold}):", count)

Number of similar pairs (distance ≤ 5): 519094
